## Environment Setup and Package Installation

This section sets up the Hugging Face cache directory and a custom Python package installation path within your Google Drive. It also installs necessary libraries, ensuring specific versions for `numpy` and `protobuf` to resolve potential compatibility issues.

Packages will only be installed if they are not already found in the environment.

In [1]:
import os
from google.colab import drive

print("Mounting Google Drive...")
drive.mount('/content/drive', force_remount=True)
print("Google Drive mounted.")

Mounting Google Drive...
Mounted at /content/drive
Google Drive mounted.


In [3]:
import os
import sys

hf_cache_dir = '/content/drive/MyDrive/hf_cache'
os.makedirs(hf_cache_dir, exist_ok=True)
os.environ['HF_HOME'] = hf_cache_dir
print(f"Hugging Face cache directory set to: {os.environ['HF_HOME']}. Models will be cached here.")

path_to_packages = '/content/drive/MyDrive/colab_packages'
os.makedirs(path_to_packages, exist_ok=True)
if path_to_packages not in sys.path:
    sys.path.insert(0, path_to_packages)
print(f"Pip installation target set to: {path_to_packages}")

Hugging Face cache directory set to: /content/drive/MyDrive/hf_cache. Models will be cached here.
Pip installation target set to: /content/drive/MyDrive/colab_packages


In [4]:
import importlib.util

# Check if sam_audio is already installed
if importlib.util.find_spec("sam_audio") is None:
    print("Installing necessary packages...")
    !pip install -q --target={path_to_packages} 'git+https://github.com/facebookresearch/sam-audio.git'
    !pip install -q --target={path_to_packages} huggingface_hub torchaudio
    # Install specific versions of numpy and protobuf to prevent conflicts
    !pip install -q --target={path_to_packages} "numpy==2.0.2" "protobuf>=5.26.1,<6"
    print("Packages installed.")
else:
    print("sam_audio and its dependencies appear to be installed.")

sam_audio and its dependencies appear to be installed.


In [ ]:
import os; os.kill(os.getpid(), 9)

In [5]:
from sam_audio.model.base import BaseModel
from sam_audio import SAMAudio, SAMAudioProcessor

_orig = BaseModel._from_pretrained.__func__

@classmethod
def _patched(cls, *, proxies=None, resume_download=False, **kwargs):
    return _orig(cls, proxies=proxies, resume_download=resume_download, **kwargs)

BaseModel._from_pretrained = _patched

/content/drive/MyDrive/colab_packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/content/drive/MyDrive/colab_packages/torch/jit/_script.py:365: FutureWarning: `torch.jit.script_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [ ]:
import os
import torch
import torchaudio
import numpy as np
from google.colab import userdata
from huggingface_hub import login, HfApi

print("NumPy Version:", np.__version__)

# 1. Retrieve Hugging Face token from Colab Secrets
hf_token = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token
login(token=hf_token)

# 2. Authenticate
login(token=hf_token)

# 3. Verify access
try:
    user_info = HfApi().whoami()
    print(f"\n✅ Success! Logged in as: {user_info['name']}")
    print(f"🔑 Token permission level: {user_info.get('auth', {}).get('accessToken', {}).get('role', 'unknown')}")
except Exception as e:
    print(f"\n❌ Authentication Error: {e}")
    print("Ensure 'HF_TOKEN' is added to Colab Secrets with 'Notebook access' enabled.")

# 4. Load SAM-Audio Model and Processor
from sam_audio import SAMAudio, SAMAudioProcessor

MODEL_ID = "facebook/sam-audio-small"

print(f"\nLoading {MODEL_ID}...")
processor = SAMAudioProcessor.from_pretrained(MODEL_ID)
model = SAMAudio.from_pretrained(MODEL_ID)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.eval().to(device)

print(f"✅ SAM-Audio successfully initialized on: {device.upper()}")

# 5. Przykładowe użycie (Audio + Prompt tekstowy)
import torch

# Utworzenie krótkiego testowego sygnału audio (1 sekunda szumu / ciszy)
sample_rate = 16000
duration = 1.0
dummy_audio = torch.randn(1, int(sample_rate * duration))

# Przygotowanie wejścia (przykład z promptem tekstowym)
text_prompt = "speech"
inputs = processor(
    audios=dummy_audio,
    sampling_rate=sample_rate,
    text=text_prompt,
    return_tensors="pt"
).to(device)

# Predykcja maski audio
with torch.no_grad():
    outputs = model(**inputs)

print("✅ Przetwarzanie zakończone sukcesem!")
print("Kształt wyjściowy masek (masks shape):", outputs.masks.shape)

NumPy Version: 2.1.3


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.



✅ Success! Logged in as: Glitch55
🔑 Token permission level: fineGrained

Loading facebook/sam-audio-small...


config.json:   0%|          | 0.00/2.25k [00:00<?, ?B/s]

/content/drive/MyDrive/colab_packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/99 [00:00<?, ?it/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

  7%|▋         | 329M/4.47G [00:17<00:55, 80.3MB/s]